# 04 · Federated learning on heterogeneous hospitals

Real hospitals do not hold random samples of one dataset. They buy different
ECG machines and serve different patients. This notebook measures what that
**heterogeneity** costs, with three ways of building hospitals from PTB-XL:

| Partition | Hospitals | What differs between them |
|---|---|---|
| By recording site | 4 (sites 0, 1, 2, rest pooled) | patient mix, mildly |
| By ECG device | 8 (7 devices, rest pooled) | hardware *and* patient mix |
| Dirichlet label skew, alpha 0.3 | 10 | label mix, strongly (synthetic) |

Each is trained with FedAvg and with **FedProx**, which adds
`mu / 2 * ||w - w_global||^2` to every hospital's loss (mu = 0.01). The
proximal term limits how far a hospital with an unusual label mix can drag its
local model within one round.

Prerequisite:

```bash
for p in site device dirichlet; do
  for s in fedavg fedprox; do uv run python scripts/train_federated.py --config ${s}_$p.yaml; done
done
```

In [ ]:
%matplotlib inline
import pandas as pd

from fedecg.paths import TABLES_DIR
from fedecg.viz import plot_curves

experiments = pd.read_csv(TABLES_DIR / "experiments.csv")
baseline = experiments.set_index("run").loc["centralized"]


def history(run: str) -> pd.DataFrame:
    return pd.read_csv(TABLES_DIR / f"{run}_history.csv")


def with_gap(rows: pd.DataFrame) -> pd.DataFrame:
    """Add the AUROC gap to the centralized baseline, the number each phase is about."""
    rows = rows.assign(gap=rows["macro_auroc"] - baseline["macro_auroc"])
    return rows.set_index("setting")

## How different are the hospitals?

Each training script writes the size and label prevalence (% of that
hospital's ECGs) of every client. Compare each row with the dataset as a
whole: NORM 44%, MI 26%, STTC 25%, CD 23%, HYP 12%.

In [ ]:
for partition in ["site", "device", "dirichlet"]:
    clients = pd.read_csv(TABLES_DIR / f"fedavg_{partition}_clients.csv", index_col="client")
    display(clients.style.set_caption(partition).format("{:.1f}", subset=clients.columns[1:]))

The device partition is the realistic one: `CS-12   E` holds 82% normal ECGs
and `CS100    3` only 30%. The Dirichlet partition is deliberately extreme,
with hospitals that almost never see some classes.

## The cost of heterogeneity, and what FedProx recovers

In [ ]:
phase5 = experiments[experiments["phase"] == 5]
columns = ["partition", "algorithm", "n_clients", "macro_auroc", "gap", "best_step"]
with_gap(phase5)[columns].round(4)

In [ ]:
paired = phase5.pivot(index="partition", columns="algorithm", values="macro_auroc")
paired["fedprox_minus_fedavg"] = paired["fedprox"] - paired["fedavg"]
paired.round(4)

In [ ]:
curves = {"centralized (per epoch)": history("centralized")}
curves |= {row.setting: history(row.run) for row in phase5.itertuples()}
fig = plot_curves(curves, step_label="epoch (centralized) or round (federated)")

In [ ]:
per_class = phase5.set_index("setting")[[f"auroc_{c}" for c in ["NORM", "MI", "STTC", "CD", "HYP"]]]
per_class.sub(baseline[per_class.columns].astype(float)).round(4)

The last table is each run's per-class AUROC minus the centralized baseline's,
so negative numbers are the classes that heterogeneity hurts most.

## What this means

- **Realistic heterogeneity costs nothing beyond decentralization itself.**
  Compare runs with a similar number of hospitals: by site (4 hospitals)
  scores 0.913 against 0.910 for 5 IID hospitals, and by device (8) scores
  0.911, above both 5 and 10 IID hospitals (0.901). The device partition has
  hospitals with 30% and with 82% normal ECGs, and FedAvg does not suffer for
  it. Unequal sizes help: the largest hospitals dominate the average.
- **Synthetic label skew does cost:** Dirichlet(0.3) across 10 hospitals scores
  0.884, 0.017 below 10 IID hospitals. MI, CD and HYP lose the most (0.04 to
  0.05 against the baseline), the classes some skewed hospitals almost never
  see.
- **FedProx does not help here.** With mu = 0.01 it is 0.003 to 0.004 below
  FedAvg on every partition. A validation-only check on the label-skew
  partition (with the earlier 32-channel network) found mu = 0.1 and 1.0 no
  better. With one local epoch per round, clients do not drift far enough for
  the proximal term to pay off; it mainly slows local progress.
- **The expensive part of federating PTB-XL is fragmentation, not
  heterogeneity:** the number of hospitals moves AUROC more than how different
  they are, unless the label skew is extreme. The same held before the
  baseline was tuned.